In [4]:
import pandas as pd
import numpy as np
import os
# import gc
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil
from sqlalchemy import create_engine
from sqlalchemy import URL


In [8]:
df_database = pd.read_parquet("../planning_data/database/database_cleaned_long.parquet", engine="pyarrow")
df_database


,Time_Bucket,Location,Count
0,2020-02-24 00:00:00,Z01_T4-ChurchSt_RevDoor_IN,1.0
1,2020-02-24 00:30:00,Z01_T4-ChurchSt_RevDoor_IN,0.0
2,2020-02-24 01:00:00,Z01_T4-ChurchSt_RevDoor_IN,0.0
3,2020-02-24 01:30:00,Z01_T4-ChurchSt_RevDoor_IN,2.0
4,2020-02-24 02:00:00,Z01_T4-ChurchSt_RevDoor_IN,1.0
...,...,...,...
15968251,2025-02-28 21:30:00,Z29_C2_SWOculus_HubElev_INTOELEV,2.0
15968252,2025-02-28 22:00:00,Z29_C2_SWOculus_HubElev_INTOELEV,0.0
15968253,2025-02-28 22:30:00,Z29_C2_SWOculus_HubElev_INTOELEV,1.0
15968254,2025-02-28 23:00:00,Z29_C2_SWOculus_HubElev_INTOELEV,0.0


In [9]:
# Split into separate columns
df_database['date'] = df_database['Time_Bucket'].dt.date#.astype('datetime64[ns]')
df_database['time'] = df_database['Time_Bucket'].dt.time
df_database = df_database.drop(columns=['Time_Bucket'])

df_database.columns = [c.lower() for c in df_database.columns] #postgres doesn't like capitals or spaces
df_database.columns = df_database.columns.str.replace('[^a-zA-Z0-9]', '').str.lower()
df_database.columns = df_database.columns.str.replace("/", '_')
df_database.columns = df_database.columns.str.replace(r' ', '_')
df_database.columns = df_database.columns.str.replace('-', '')
df_database.columns = df_database.columns.str.strip('_')
df_database

,location,count,date,time
0,Z01_T4-ChurchSt_RevDoor_IN,1.0,2020-02-24,00:00:00
1,Z01_T4-ChurchSt_RevDoor_IN,0.0,2020-02-24,00:30:00
2,Z01_T4-ChurchSt_RevDoor_IN,0.0,2020-02-24,01:00:00
3,Z01_T4-ChurchSt_RevDoor_IN,2.0,2020-02-24,01:30:00
4,Z01_T4-ChurchSt_RevDoor_IN,1.0,2020-02-24,02:00:00
...,...,...,...,...
15968251,Z29_C2_SWOculus_HubElev_INTOELEV,2.0,2025-02-28,21:30:00
15968252,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,2025-02-28,22:00:00
15968253,Z29_C2_SWOculus_HubElev_INTOELEV,1.0,2025-02-28,22:30:00
15968254,Z29_C2_SWOculus_HubElev_INTOELEV,0.0,2025-02-28,23:00:00


In [10]:
url_object = URL.create(
    "postgresql",
    username="postgres",
    password="DeathLife!@#",  # plain (unescaped) text
    host="localhost",
    database="Xovis_Counts",
)
engine = create_engine(url_object)

In [11]:
df_database.to_sql("database_cleaned", engine, if_exists='replace', index=False)


256